# V9.1: MambaConcatFusion (Concatenate+Scan)

Replace cross-attention (SeqCA) with Mamba-native fusion:
- Compress text into 8 prompt tokens
- Concatenate with image tokens
- Unified Mamba scan (selective scan does implicit cross-modal interaction)

Reference: MFuser (CVPR 2025 Highlight)

| Version | Fusion | Mean Dice | Text Delta |
|---------|--------|-----------|------------|
| V5.0 | SeqCA | 0.8479 | +0.55% |
| V8.0 | SeqCA + pretrain | 0.8753 | 0.00% |
| V9.0 | SeqCA + freeze/dropout/align | 0.8723 | -0.02% |
| **V9.1** | **ConcatScan + pretrain** | **?** | **?** |


In [2]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi 2>/dev/null || echo 'No GPU'

!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel pyyaml tqdm scipy

import os, subprocess, zipfile, time, shutil, glob, threading
REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)

os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    for attempt in range(1, 4):
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True)
        if ret.returncode == 0:
            break
        print(f'Clone attempt {attempt} failed')
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        time.sleep(5 * attempt)
    else:
        raise RuntimeError('Clone failed')
    os.chdir(REPO_DIR)

# BraTS2020 + TextBraTS
DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
if os.path.exists(ET_CACHE):
    with zipfile.ZipFile(ET_CACHE, 'r') as zf:
        zf.extractall(DATA_DIR)
print(f'BraTS2020 data: {len([d for d in os.listdir(DATA_DIR) if d.startswith("BraTS")])} cases')

def sync_and_tag(tag):
    local_ckpt = os.path.join(REPO_DIR, 'checkpoints')
    if not os.path.exists(local_ckpt):
        return
    for f in glob.glob(os.path.join(local_ckpt, '*.pth')):
        shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
    best = os.path.join(local_ckpt, 'best.pth')
    if os.path.exists(best):
        shutil.copy2(best, os.path.join(DRIVE_CKPT, f'best_{tag}.pth'))
        print(f'Tagged: best_{tag}.pth')
    last = os.path.join(local_ckpt, 'last.pth')
    if os.path.exists(last):
        shutil.copy2(last, os.path.join(DRIVE_CKPT, f'last_{tag}.pth'))
    print(f'Synced to {DRIVE_CKPT}')

# Background auto-sync
_sync_stop = threading.Event()
_sync_track = {}

def _file_is_stable(path, wait=3):
    try:
        s1 = os.path.getsize(path)
        time.sleep(wait)
        s2 = os.path.getsize(path)
        return s1 == s2 and s1 > 0
    except OSError:
        return False

def _bg_sync_loop():
    local_ckpt = os.path.join(REPO_DIR, 'checkpoints')
    while not _sync_stop.is_set():
        _sync_stop.wait(120)
        if _sync_stop.is_set():
            break
        try:
            files = glob.glob(os.path.join(local_ckpt, '*.pth'))
            synced = 0
            for f in files:
                mt = os.path.getmtime(f)
                name = os.path.basename(f)
                if name not in _sync_track or _sync_track[name] < mt:
                    if not _file_is_stable(f):
                        continue
                    shutil.copy2(f, os.path.join(DRIVE_CKPT, name))
                    _sync_track[name] = mt
                    synced += 1
            if synced > 0:
                print(f'[AutoSync] {synced} checkpoint(s) synced to Drive')
        except Exception as e:
            print(f'[AutoSync] Warning: {e}')

_sync_thread = threading.Thread(target=_bg_sync_loop, daemon=True)
_sync_thread.start()
print('Background auto-sync started (every 2 min)')
print('Setup complete')


Mounted at /content/drive
Fri Apr  3 16:12:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

## V9.1 Training: ConcatScan Fine-tune

From V8.0 Stage 1 checkpoint, fine-tune with MambaConcatFusion.


In [3]:
import os, glob
os.chdir(REPO_DIR)

os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

# Only use stage1-specific checkpoints, never shared names
STAGE1_CKPT = os.path.join(DRIVE_CKPT, 'best_V8.0_stage1.pth')
if not os.path.exists(STAGE1_CKPT):
    STAGE1_CKPT = os.path.join(DRIVE_CKPT, 'last_stage1.pth')
assert os.path.exists(STAGE1_CKPT), (
    f'Stage 1 checkpoint not found! Need best_V8.0_stage1.pth or last_stage1.pth in {DRIVE_CKPT}'
)

for f in glob.glob(os.path.join(REPO_DIR, 'checkpoints', '*.pth')):
    os.remove(f)

print(f'V9.1: ConcatScan fine-tune from {STAGE1_CKPT}')
!python -u train.py \
    --config configs/autoresearch/V9.1_concat_scan.yaml \
    --resume "{STAGE1_CKPT}" \
    --reset-optimizer \
    --no-text-ratio 0.15 \
    --grad-accum 2

sync_and_tag('V9.1')
print('V9.1 complete!')


V9.1: ConcatScan fine-tune from /content/drive/MyDrive/TextMamba3D/checkpoints/best_V8.0_stage1.pth
2026-04-03 16:16:43.901797: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-03 16:16:43.910109: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775233003.919754    9035 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775233003.922977    9035 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775233003.931170    9035 compu

In [4]:
# === Emergency Sync ===
import shutil, glob, os, subprocess

local_ckpt = os.path.join(REPO_DIR, 'checkpoints')
files = glob.glob(os.path.join(local_ckpt, '*.pth'))
if not files:
    print('No local checkpoints')
else:
    for f in sorted(files):
        name = os.path.basename(f)
        shutil.copy2(f, os.path.join(DRIVE_CKPT, name))
    subprocess.run(['sync'], check=True)
    print(f'{len(files)} files synced')


3 files synced


## Evaluation


In [5]:
import subprocess, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_V9.1.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, 'last.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Evaluating: {ckpt}')

CONFIG = 'configs/autoresearch/V9.1_concat_scan.yaml'

for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    print()
    print('=' * 60)
    print(name)
    print('=' * 60)
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', CONFIG,
           '--checkpoint', ckpt,
           '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    print(ret.stdout)
    if ret.returncode != 0:
        print(f'ERROR: {ret.stderr[-500:]}')

print()
print('Comparison:')
print('  V8.0: Mean=0.8753, text_delta=0.00%')
print('  V9.0: Mean=0.8723, text_delta=-0.02%')
print('  V5.0: Mean=0.8479, text_delta=+0.55%')
print('  TextBraTS SOTA: text_delta=+1.5%')


Evaluating: /content/drive/MyDrive/TextMamba3D/checkpoints/best_V9.1.pth

text+TTA
[AutoSync] 1 checkpoint(s) synced to Drive
Loaded checkpoint: epoch=1, best_dice=0.9009343429848
TextBraTS test: 95 samples

Evaluating 95 cases (test split)
Sliding window: patch=(128, 128, 128), overlap=0.5, text=True
TTA: 8-fold flip ensemble ENABLED

  BraTS20_Training_328: Dice=0.6989 (ET=0.3796, TC=0.7877, WT=0.9293) HD95_ET=50.80
  BraTS20_Training_028: Dice=0.7876 (ET=0.6497, TC=0.8658, WT=0.8472) HD95_ET=1.73
  BraTS20_Training_289: Dice=0.5954 (ET=0.0000, TC=0.8398, WT=0.9464) HD95_ET=nan
  BraTS20_Training_231: Dice=0.9444 (ET=0.9274, TC=0.9433, WT=0.9624) HD95_ET=1.00
  BraTS20_Training_261: Dice=0.7238 (ET=0.4943, TC=0.7264, WT=0.9507) HD95_ET=5.74
  BraTS20_Training_163: Dice=0.9218 (ET=0.8515, TC=0.9397, WT=0.9743) HD95_ET=1.00
  BraTS20_Training_345: Dice=0.8588 (ET=0.8476, TC=0.8319, WT=0.8969) HD95_ET=6.78
  BraTS20_Training_139: Dice=0.9045 (ET=0.9021, TC=0.9318, WT=0.8796) HD95_ET=1.0